# Name That LLM: Exploratory Data Analysis

**Goal:** understand the data *before* modelling it. Each section answers one question
whose answer changes a modelling decision.

This notebook is part of a learning-in-public, end-to-end MLOps project:
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm). All data handling
(loading, validation, cleaning) comes from the project's own package, so this notebook sees
the data exactly the way the models will.

> **Data rule:** the competition data may not be redistributed. This notebook therefore shows
> **aggregates only** (counts, rates, short n-grams, similarities), never full passages.

## 1. Setup: what does Kaggle's environment look like?

The final submission notebook runs **without internet**, so it can only use what Kaggle's
image already ships, plus files we attach ourselves. Three facts decide how we package our
code for it:
- which of our package's less common dependencies are already installed,
- the Python version,
- the folder where Kaggle mounts the competition data.

In [ ]:
# Run this BEFORE installing anything: which of our package's less common dependencies
# does Kaggle's image already ship? The offline submission notebook must bring every
# missing one along as a wheel file, so we need this list.
import importlib.util

DEPS_TO_CHECK = ["pandera", "typeguard", "typing_inspect", "mypy_extensions"]
preinstalled = {name: importlib.util.find_spec(name) is not None for name in DEPS_TO_CHECK}
print("Preinstalled on Kaggle:", preinstalled)

In [ ]:
# Install the project's package from a git TAG, not from "main". A tag never moves, so this
# notebook always runs against exactly the code it was written for (reproducibility).
%pip install -q "name-that-llm @ git+https://github.com/Ryqn0/name-that-llm@v0.1.0"

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import sklearn

from name_that_llm.data import LAB_OF, LABELS, find_data_dir, load_train

print("Python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

# Where did Kaggle mount the data? Our loader tries several known paths. Printing the
# one it found tells us which is real (Review Focus 4 in the plan).
if os.path.isdir("/kaggle/input"):
    print("\n/kaggle/input contains:", sorted(os.listdir("/kaggle/input")))
print("data folder used:", find_data_dir())

# load_train() reads the file with ids kept as strings, applies NFC cleaning, and validates
# the schema. If the data broke any rule (unknown label, duplicate prompt/model pair, ...),
# this line raises a precise error instead of letting a problem slip through silently.
train = load_train()
print("train:", train.shape)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# One visual system for every chart. The palette comes from a colorblind-checked reference
# set, and colors carry MEANING, never decoration:
#   - categorical (identity):  one fixed color per LAB, always in the same order
#   - sequential (magnitude):  one hue, light = low, dark = high
#   - diverging (vs. average): blue <- neutral gray -> red
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS = "#e1e0d9", "#c3c2b7"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.labelcolor": INK_2,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "axes.edgecolor": AXIS,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,  # quiet grid lines
        "axes.spines.top": False,
        "axes.spines.right": False,  # less ink, same info
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "figure.dpi": 110,
    }
)

LABS = sorted(set(LAB_OF.values()))  # anthropic, google, moonshot, openai, qwen, zai
# strict=True: raise if the two lists ever differ in length (6 labs <-> 6 colors).
LAB_COLORS = dict(
    zip(LABS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"], strict=True)
)
SERIES_1 = "#2a78d6"  # the color for a single-series chart
SEQUENTIAL = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING = LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#f09a99", "#b83131"]
)

# Models grouped by lab, so same-lab models sit next to each other in every chart.
MODEL_ORDER = sorted(LABELS, key=lambda m: (LAB_OF[m], m))


def tagged(model):
    # Put the lab in the label TEXT, so identity never depends on color alone (accessibility).
    return f"{model} ({LAB_OF[model]})"


def lab_legend(ax):
    handles = [plt.Rectangle((0, 0), 1, 1, color=LAB_COLORS[lab]) for lab in LABS]
    ax.legend(
        handles,
        LABS,
        ncol=6,
        frameon=False,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.0),
        fontsize=9,
    )

## 2. Shape and balance: is the data what we think it is?

Two things to confirm:
- **Balance.** If every model has the same number of passages, balanced accuracy (the
  competition metric) behaves like plain accuracy, and no class needs re-weighting.
- **One answer per model per prompt.** This structure matters twice. Validation must hold
  out whole prompts, because test prompts are new. And on the hidden test we may be able to
  solve each prompt as a small "who wrote which answer" puzzle (idea I-001).

In [ ]:
rows_per_model = train["model"].value_counts().reindex(MODEL_ORDER)
print("Rows per model: min", rows_per_model.min(), "/ max", rows_per_model.max())

rows_per_prompt = train.groupby("prompt_id").size()
print("Prompts:", train["prompt_id"].nunique())
print("Rows per prompt -> number of prompts:", rows_per_prompt.value_counts().to_dict())

# The key structural fact: within one prompt, does any model ever appear twice?
max_repeat = train.groupby(["prompt_id", "model"]).size().max()
print("Most answers by one model to one prompt:", max_repeat)

# Fail loudly if the data ever stops matching what the project is built on.
assert rows_per_prompt.value_counts().to_dict() == {18: 149, 17: 1}, "prompt structure changed"
assert max_repeat == 1
rows_per_model.rename("passages").to_frame().T  # small table view instead of a flat bar chart

## 3. Length: can length alone tell the models apart?

If some models were much wordier than others, a single number (word count) would already be
a strong feature. The box plots show each model's spread. If the boxes overlap a lot, length
is a weak signal and the real fingerprints are elsewhere.

In [ ]:
words = train["text"].str.split().str.len()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(words, bins=40, color=SERIES_1, edgecolor=SURFACE, linewidth=0.5)
ax.set_title("Words per passage (all 2,699 passages)")
ax.set_xlabel("words")
ax.set_ylabel("passages")
plt.show()

fig, ax = plt.subplots(figsize=(11, 4.5))
boxes = ax.boxplot(
    [words[train["model"] == m] for m in MODEL_ORDER],
    patch_artist=True,
    widths=0.6,
    medianprops={"color": INK, "linewidth": 1.5},
    whiskerprops={"color": MUTED},
    capprops={"color": MUTED},
    flierprops={
        "marker": "o",
        "markersize": 3,
        "markerfacecolor": MUTED,
        "markeredgecolor": "none",
    },
)
for box, model in zip(boxes["boxes"], MODEL_ORDER, strict=True):
    box.set_facecolor(LAB_COLORS[LAB_OF[model]])
    box.set_edgecolor(SURFACE)
ax.set_xticks(range(1, len(MODEL_ORDER) + 1), MODEL_ORDER, rotation=60, ha="right")
ax.set_ylabel("words per passage")
ax.set_title("Passage length per model", pad=28)
lab_legend(ax)
plt.show()

words.groupby(train["model"]).median().reindex(MODEL_ORDER).rename("median words").to_frame().T

## 4. Punctuation habits: a style fingerprint

The host removed formatting (newlines, bold, lists), but punctuation survived, and models
have strong habits: some love em dashes, some use curly quotes, some never use semicolons.

For each model we compute a **pooled rate per 1,000 characters**: all marks of that kind,
divided by all characters the model wrote. Pooling is robust to short passages. Each column
is then turned into a **z-score**, i.e. how far the model is from the 18-model average for
that mark, so rare marks and common marks share one color scale.

In [ ]:
import re

MARKS = {
    "em dash —": "—",
    "en dash –": "–",
    "hyphen -": "-",
    "semicolon ;": ";",
    "colon :": ":",
    "paren (": "(",
    "paren )": ")",
    'straight "': '"',
    "straight '": "'",
    "curly “": "“",
    "curly ”": "”",
    "curly ‘": "‘",
    "curly ’": "’",
    "ellipsis …": "…",
    "exclamation !": "!",
    "question ?": "?",
}
# str.count() takes a regular expression, so marks like "(" or "?" must be escaped.
counts = pd.DataFrame({name: train["text"].str.count(re.escape(ch)) for name, ch in MARKS.items()})
counts["digits 0-9"] = train["text"].str.count(r"\d")
chars = train["text"].str.len()

per_model = counts.groupby(train["model"]).sum().div(chars.groupby(train["model"]).sum(), axis=0)
per_model = (per_model * 1000).reindex(MODEL_ORDER)

# z-score per column. A mark nobody uses has std 0, so fill with 0 (= exactly average).
z = ((per_model - per_model.mean()) / per_model.std(ddof=0)).fillna(0)
limit = float(np.abs(z.to_numpy()).max())

fig, ax = plt.subplots(figsize=(11, 7))
image = ax.imshow(
    z.to_numpy(),
    cmap=DIVERGING,
    aspect="auto",
    norm=TwoSlopeNorm(vcenter=0, vmin=-limit, vmax=limit),
)
ax.set_xticks(range(z.shape[1]), z.columns, rotation=60, ha="right")
ax.set_yticks(range(len(MODEL_ORDER)), [tagged(m) for m in MODEL_ORDER])
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8, label="z-score vs. the 18-model average")
ax.set_title("Punctuation per model: above (red) or below (blue) average")
plt.show()

per_model.round(2)  # table view: the raw rates behind the colors

## 5. Discriminative words: what does each lab say more often?

The **chi-squared (χ²) test** measures how unevenly a word (or two-word phrase) is spread
between one lab and all the others. Every model answered the *same* prompts, so topic words
cancel out, and what remains is mostly wording habits. χ² doesn't say in which direction a
word is uneven, so we keep only words the lab uses **more** than the rest.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

# min_df=5: ignore rare n-grams, which are mostly topic words from a single prompt.
ngram_vec = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_words = ngram_vec.fit_transform(train["text"])
vocab = ngram_vec.get_feature_names_out()
lab_of_row = train["model"].map(LAB_OF).to_numpy()

top_ngrams = {}
for lab in LABS:
    in_lab = lab_of_row == lab
    scores, _ = chi2(X_words, in_lab)
    used_more = (
        np.asarray(X_words[in_lab].mean(axis=0)).ravel()
        > np.asarray(X_words[~in_lab].mean(axis=0)).ravel()
    )
    scores = np.where(used_more, np.nan_to_num(scores), 0.0)
    top_ngrams[lab] = vocab[np.argsort(scores)[::-1][:10]]

pd.DataFrame(top_ngrams, index=pd.RangeIndex(1, 11, name="rank"))

## 6. Model similarity: which models write alike?

We turn every passage into a vector of **character 1–5-gram** weights (TF-IDF). Character
n-grams capture punctuation, spacing and word endings, not just whole words. Averaging all
of one model's vectors gives its **centroid**, its "average style". Since every model
answered the same prompts, topic averages out of the centroids. Cosine similarity between
centroids then shows which models are hard to tell apart, and the dendrogram groups them
into families.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# lowercase=False: casing is part of style. sublinear_tf: damp words repeated many times.
char_vec = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 5),
    lowercase=False,
    sublinear_tf=True,
    min_df=2,
    max_features=200_000,
)
X_chars = char_vec.fit_transform(train["text"])

centroids = normalize(
    np.vstack(
        [np.asarray(X_chars[(train["model"] == m).to_numpy()].mean(axis=0)) for m in MODEL_ORDER]
    )
)
similarity = centroids @ centroids.T  # rows have unit length, so this is cosine similarity

off_diagonal = similarity[~np.eye(len(MODEL_ORDER), dtype=bool)]
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(similarity, cmap=SEQUENTIAL, vmin=off_diagonal.min(), vmax=off_diagonal.max())
ax.set_xticks(range(len(MODEL_ORDER)), MODEL_ORDER, rotation=60, ha="right")
ax.set_yticks(range(len(MODEL_ORDER)), [tagged(m) for m in MODEL_ORDER])
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.8, label="cosine similarity of style centroids")
ax.set_title("How alike the 18 models write (darker = more alike)")
plt.show()

distance = np.clip(1 - similarity, 0, None)
np.fill_diagonal(distance, 0)
tree = linkage(squareform(distance, checks=False), method="average")
fig, ax = plt.subplots(figsize=(9, 5.5))
dendrogram(
    tree,
    labels=[tagged(m) for m in MODEL_ORDER],
    orientation="left",
    color_threshold=0,
    above_threshold_color=MUTED,
    ax=ax,
)
ax.set_title("Model families by character style")
ax.set_xlabel("distance (1 - cosine similarity)")
ax.grid(False)
plt.show()

most_alike = (
    pd.DataFrame(similarity, index=MODEL_ORDER, columns=MODEL_ORDER)
    .where(~np.eye(len(MODEL_ORDER), dtype=bool))
    .stack()
    .sort_values(ascending=False)
    .iloc[::2]
    .head(8)
)  # each pair appears twice
most_alike.rename("similarity").round(4).to_frame()

## 7. Label-noise check (idea I-004): can we tell gemini-3.7-flash from gemini-3.8-flash?

A forum thread reports that requests to gemini-3.7-flash may have been answered by
gemini-3.8-flash. If that's true, some "3.7" labels are really 3.8 text, and no model could
separate the two classes well. A classifier trained on only those two would score near
**0.5**, a coin flip.

We compare that pair with sibling pairs of the same family and with one cross-lab pair for
scale. Each score is a two-class logistic regression on character TF-IDF, validated with
**prompt-grouped** 5-fold CV, so it's always tested on prompts it never saw.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline


def pair_accuracy(a, b):
    # Prompt-grouped 5-fold CV accuracy of a two-class classifier: 0.5 = coin flip.
    pair = train[train["model"].isin([a, b])]
    clf = make_pipeline(
        TfidfVectorizer(
            analyzer="char", ngram_range=(1, 5), lowercase=False, sublinear_tf=True, min_df=2
        ),
        LogisticRegression(C=4.0, max_iter=2000),
    )
    scores = cross_val_score(
        clf,
        pair["text"],
        pair["model"],
        groups=pair["prompt_id"],
        cv=GroupKFold(n_splits=5),
        scoring="accuracy",
    )
    return scores.mean(), scores.std()


PAIRS = [
    ("gemini-3.7-flash", "gemini-3.8-flash"),  # the suspected pair
    ("gemini-3.6-flash", "gemini-3.8-flash"),  # sibling reference
    ("gemini-3.6-flash", "gemini-3.7-flash"),  # sibling reference
    ("gpt-5.5", "claude-opus-5"),  # cross-lab reference, for scale
]
noise = pd.DataFrame(
    [(f"{a} vs {b}", *pair_accuracy(a, b)) for a, b in PAIRS],
    columns=["pair", "accuracy", "std across folds"],
).set_index("pair")
noise.round(3)

## 8. Style-only clustering (idea I-006): do models cluster by lab once topic is removed?

**Function words** (the, of, however, which, ...) and punctuation barely depend on the topic,
which makes them classic authorship features. We describe each model by its average
function-word rates and punctuation rates, then cluster the 18 models into 6 groups. The
**adjusted Rand index** compares those clusters with the 6 labs: 1 means the clusters *are*
the labs, about 0 means they're unrelated.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score

fw_vec = CountVectorizer(vocabulary=sorted(ENGLISH_STOP_WORDS))
fw_counts = fw_vec.transform(train["text"]).toarray().astype(float)
tokens = train["text"].str.split().str.len().to_numpy()[:, None]

style = pd.concat(
    [
        pd.DataFrame(fw_counts / tokens * 1000, columns=fw_vec.get_feature_names_out()),
        counts.div(chars, axis=0) * 1000,  # punctuation rates from section 4
    ],
    axis=1,
)
style_per_model = style.groupby(train["model"].to_numpy()).mean().reindex(MODEL_ORDER)

# Standardize each feature, so frequent words don't dominate just by being frequent.
spread = style_per_model.std(ddof=0).replace(0, 1)
standardized = ((style_per_model - style_per_model.mean()) / spread).to_numpy()
style_tree = linkage(standardized, method="ward")

fig, ax = plt.subplots(figsize=(9, 5.5))
dendrogram(
    style_tree,
    labels=[tagged(m) for m in MODEL_ORDER],
    orientation="left",
    color_threshold=0,
    above_threshold_color=MUTED,
    ax=ax,
)
ax.set_title("Model families by style only (function words + punctuation)")
ax.grid(False)
plt.show()

clusters = fcluster(style_tree, t=6, criterion="maxclust")
labs_in_order = [LAB_OF[m] for m in MODEL_ORDER]
print(
    f"Adjusted Rand index, 6 style clusters vs 6 labs: "
    f"{adjusted_rand_score(labs_in_order, clusters):.2f}"
)
pd.crosstab(pd.Series(labs_in_order, name="lab"), pd.Series(clusters, name="style cluster"))

## 9. Findings

*Filled in after the run.*